# Workshop: Evaluación de Agentes con LangGraph + agentevals

En este workshop construirás un agente desde cero y lo evaluarás
usando tres tipos de evaluadores.

**Tiempo estimado:** 1.5 horas

## Lo que harás
1. Definir un agente con tools propias
2. Construir un dataset de evaluación
3. Implementar evaluadores determinísticos y LLM-as-judge
4. Analizar los resultados

## Prerrequisitos
- Completar el notebook `agent_evals.ipynb`
- API key de NVIDIA NIM o Groq

## 0. Instalación y setup

In [ ]:
!pip install langgraph langchain-openai langchain-core agentevals langsmith ddgs python-dotenv pandas pydantic

In [ ]:
import os
import json
import time
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.errors import GraphRecursionError

In [ ]:
load_dotenv()

In [ ]:
# ── Elige tu provider ──────────────────────────────────────────
# Opción 1: NVIDIA NIM
llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
    model="meta/llama-3.1-70b-instruct",
)

Opción 2: Groq (descomentar si usas Groq)
from langchain_groq import ChatGroq
llm = ChatGroq(api_key=os.environ["GROQ_API_KEY"], model="llama-3.1-70b-versatile")

In [ ]:
print("LLM listo ✅")

─────────────────────────────────────────────────────────────
PARTE 1: Define tu agente y tools (20 min)
─────────────────────────────────────────────────────────────

## 1.1 Define tus tools

Tu agente debe tener AL MENOS 2 tools.
Puedes usar herramientas de búsqueda, calculadoras, conversores, etc.

Reglas:
- Cada tool debe tener un docstring claro (el LLM lo usa para decidir cuándo llamarla)
- Deben retornar siempre un string
- Los nombres deben ser descriptivos

Ejemplo de tool ya implementada:

In [ ]:
from ddgs import DDGS

In [ ]:
@tool
def web_search(query: str) -> str:
    """Busca información actualizada en la web. Úsala para preguntas sobre
    eventos recientes, personas, empresas o cualquier dato que pueda haber cambiado."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
        if not results:
            return "No se encontraron resultados."
        return "\n\n".join(f"**{r['title']}**\n{r['body']}" for r in results)
    except Exception as e:
        return f"Error al buscar: {e}"

TODO: Define tu segunda tool aquí
Sugerencias:
- Una calculadora para operaciones matemáticas
- Un conversor de unidades (km a millas, °C a °F, etc.)
- Un generador de resúmenes
- Una tool de fechas (¿qué día es hoy? ¿cuántos días entre dos fechas?)

@tool
def mi_tool(param: str) -> str:
    """Descripción clara de qué hace esta tool y cuándo usarla."""
    # TODO: implementar
    pass

TODO (opcional): Define una tercera tool si quieres enriquecer tu agente

## 1.2 Construye el agente

Completa los TODO para ensamblar el agente con LangGraph.

In [ ]:
# TODO: Agrega tus tools a esta lista
TOOLS = [web_search]  # TODO: agregar tu segunda tool aquí

In [ ]:
llm_with_tools = llm.bind_tools(TOOLS, parallel_tool_calls=False)

In [ ]:
# TODO: Personaliza el system prompt para tu agente
# El system prompt define el comportamiento general del agente
SYSTEM_PROMPT = SystemMessage(
    content="""Eres un asistente útil.
    
TODO: Personaliza este system prompt según el dominio de tu agente.
Describe:
- Qué tipo de preguntas responde
- Cuándo debe usar cada tool
- Cómo debe responder (tono, formato, idioma)
"""
)

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
def call_model(state: AgentState) -> AgentState:
    messages = [SYSTEM_PROMPT] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

In [ ]:
def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

In [ ]:
# TODO: Agrega los nodos y aristas del grafo
# El grafo tiene dos nodos: "agent" (el LLM) y "tools" (ejecutor de tools)
# La arista condicional decide si seguir al nodo "tools" o terminar (END)
#
# graph_builder.add_node("agent", ???)       # nodo del LLM
# graph_builder.add_node("tools", ???)       # nodo ejecutor de tools
# graph_builder.set_entry_point("agent")     # siempre empieza en el agente
# graph_builder.add_conditional_edges(???)   # router: tools o END
# graph_builder.add_edge("tools", "agent")   # tras ejecutar tools → vuelve al agente
graph_builder = StateGraph(AgentState)
graph_builder.add_node("agent", call_model)
graph_builder.add_node("tools", ToolNode(TOOLS))
graph_builder.set_entry_point("agent")
graph_builder.add_conditional_edges("agent", should_continue)
graph_builder.add_edge("tools", "agent")

In [ ]:
agent = graph_builder.compile()
print("Agente compilado ✅")

## 1.3 Prueba manual del agente

Antes de evaluar, asegúrate de que el agente funciona correctamente.
Prueba al menos 3 preguntas diferentes.

In [ ]:
# TODO: reemplaza con preguntas relevantes para tu agente
test_questions = [
    "¿Cuánto es 15% de 340?",          # TODO: cambiar
    "¿Quién ganó el último mundial?",   # TODO: cambiar
    "¿Cuál es la capital de Japón?",    # TODO: cambiar
]

In [ ]:
for q in test_questions:
    result = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config={"recursion_limit": 10}
    )
    print(f"\nQ: {q}")
    print(f"A: {result['messages'][-1].content[:200]}")
    time.sleep(3)

─────────────────────────────────────────────────────────────
PARTE 2: Define el dataset de evaluación (20 min)
─────────────────────────────────────────────────────────────

## 2.1 Estructura del dataset

Cada ejemplo necesita:
- question: la pregunta que le harás al agente
- expected_answer: la respuesta correcta (para el juez)
- expected_tools: lista de tools que DEBE llamar ([] si no necesita ninguna)

Criterios para un buen dataset:
- Variedad: incluye preguntas que requieren tools y preguntas que no
- Claridad: la expected_answer debe ser verificable
- Cobertura: prueba cada tool al menos 2 veces
- Dificultad: mezcla preguntas fáciles y difíciles

TODO: Define tu dataset con MÍNIMO 8 ejemplos
- Al menos 3 que requieran web_search  (incluir expected_content con 2-4 términos)
- Al menos 3 que requieran tu segunda tool (expected_content = [])
- Al menos 2 que NO requieran ninguna tool (expected_content = [])

expected_content: términos clave que deben aparecer en los resultados
de web_search. Solo aplica cuando expected_tools incluye "web_search".
Para las demás tools, usa [].

In [ ]:
eval_dataset = [
    # Ejemplo sin tools
    {
        "question": "¿Cuántos lados tiene un hexágono?",
        "expected_answer": "Un hexágono tiene 6 lados",
        "expected_tools": [],
        "expected_content": [],
    },
    # TODO: Agrega tus ejemplos
    # Ejemplo con web_search:
    # {
    #     "question": "¿Quién es el CEO actual de NVIDIA?",
    #     "expected_answer": "Jensen Huang es el CEO de NVIDIA",
    #     "expected_tools": ["web_search"],
    #     "expected_content": ["Jensen Huang", "NVIDIA", "CEO"],
    # },
    # Ejemplo con tu segunda tool:
    # {
    #     "question": "¿Cuánto es el 18% de 450?",
    #     "expected_answer": "El 18% de 450 es 81",
    #     "expected_tools": ["calculator"],
    #     "expected_content": [],
    # },
]

In [ ]:
# Validación básica del dataset
assert len(eval_dataset) >= 8, "❌ El dataset debe tener al menos 8 ejemplos"

In [ ]:
for item in eval_dataset:
    assert "expected_content" in item, f"❌ Falta 'expected_content' en: {item['question']}"
    if "web_search" in item["expected_tools"]:
        assert len(item["expected_content"]) > 0, (
            f"❌ expected_content no puede estar vacío para preguntas con web_search: {item['question']}"
        )

In [ ]:
tool_counts = {}
for item in eval_dataset:
    for t in item["expected_tools"]:
        tool_counts[t] = tool_counts.get(t, 0) + 1

In [ ]:
no_tool_count = sum(1 for item in eval_dataset if item["expected_tools"] == [])
search_count = tool_counts.get("web_search", 0)

In [ ]:
print(f"✅ Dataset válido: {len(eval_dataset)} ejemplos")
print(f"   Sin tools: {no_tool_count}")
print(f"   Con web_search: {search_count} (con expected_content)")
print(f"   Por tool: {tool_counts}")

─────────────────────────────────────────────────────────────
PARTE 3: Implementa los evaluadores (30 min)
─────────────────────────────────────────────────────────────

## 3.1 Helper: extraer trayectoria

Esta función ya está implementada — extrae la trayectoria completa
del agente en el formato que espera agentevals.

In [ ]:
def run_agent_and_get_trajectory(question: str) -> dict:
    result = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"recursion_limit": 10}
    )
    messages = result["messages"]
    trajectory = [{"role": "user", "content": question}]
    tools_called = []

    for msg in messages:
        msg_type = type(msg).__name__
        if msg_type == "AIMessage":
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                trajectory.append({
                    "role": "assistant",
                    "content": msg.content or "",
                    "tool_calls": [
                        {"function": {"name": tc["name"], "arguments": json.dumps(tc["args"])}}
                        for tc in msg.tool_calls
                    ]
                })
                tools_called.extend([tc["name"] for tc in msg.tool_calls])
            else:
                trajectory.append({"role": "assistant", "content": msg.content})
        elif msg_type == "ToolMessage":
            trajectory.append({"role": "tool", "content": str(msg.content)})

    return {
        "answer": messages[-1].content,
        "trajectory": trajectory,
        "tools_called": tools_called,
    }

## 3.2 Evaluador 1: Tool use (determinístico)

TODO: Completa la función de evaluación de tool use.
La función recibe las tools que el agente llamó y las que se esperaban,
y retorna un score entre 0.0 y 1.0.

In [ ]:
def evaluate_tool_use(tools_called: list, expected_tools: list) -> dict:
    """
    Evalúa si el agente usó las tools correctas.
    
    Criterios:
    - Si expected_tools == []: el agente NO debe llamar ninguna tool
    - Si expected_tools != []: el agente DEBE llamar todas las tools esperadas
    
    Retorna: {"score": float, "comment": str}
    """
    called_set = set(tools_called)
    expected_set = set(expected_tools)

    # TODO: implementa la lógica de scoring
    # Pista: usa called_set y expected_set para calcular el score
    # score = ???
    # comment = ???

    raise NotImplementedError("TODO: implementa evaluate_tool_use")

In [ ]:
# Prueba tu evaluador antes de continuar
assert evaluate_tool_use([], [])["score"] == 1.0, "Caso 1 fallido"
assert evaluate_tool_use(["web_search"], ["web_search"])["score"] == 1.0, "Caso 2 fallido"
assert evaluate_tool_use([], ["web_search"])["score"] == 0.0, "Caso 3 fallido"
assert evaluate_tool_use(["web_search"], [])["score"] == 0.0, "Caso 4 fallido"
print("✅ evaluate_tool_use correcta")

## 3.2 Evaluador 2: Retrieval quality (determinístico)

Mide si los resultados de web_search contienen los términos esperados.
Opera sobre los mensajes role="tool" de la trayectoria.

Métrica: Recall@expected_content
- score = 1.0  → todos los términos encontrados
- score = 0.5  → solo la mitad → respuesta puede ser incompleta
- score = 0.0  → no llamó la tool o resultados irrelevantes
- score = None → no aplica (pregunta sin web_search)

In [ ]:
def evaluate_retrieval(trajectory: list, expected_content: list) -> dict:
    if not expected_content:
        return {"score": None, "comment": "N/A: no requiere búsqueda"}

    tool_results = " ".join(
        msg["content"] for msg in trajectory if msg["role"] == "tool"
    ).lower()

    if not tool_results:
        return {"score": 0.0, "comment": "❌ No se llamó ninguna tool"}

    found = [term for term in expected_content if term.lower() in tool_results]
    score = len(found) / len(expected_content)

    return {
        "score": score,
        "found": found,
        "missing": [t for t in expected_content if t.lower() not in tool_results],
        "comment": f"{len(found)}/{len(expected_content)} términos encontrados: {found}",
    }

In [ ]:
# Prueba con una pregunta que use web_search
search_sample = next(
    (item for item in eval_dataset if "web_search" in item.get("expected_tools", [])),
    None
)
if search_sample:
    search_result = run_agent_and_get_trajectory(search_sample["question"])
    print("Retrieval score:", evaluate_retrieval(
        search_result["trajectory"], search_sample["expected_content"]
    ))
else:
    print("⚠️ Agrega al menos un ejemplo con web_search para probar este evaluador")

## 3.3 Evaluador 3: Trajectory match (agentevals)

Este evaluador compara la trayectoria del agente con una de referencia.

In [ ]:
from agentevals.trajectory.match import create_trajectory_match_evaluator

In [ ]:
# TODO: Elige el trajectory_match_mode más apropiado para tu caso
# - "strict": mismo orden, mismas tools
# - "unordered": mismas tools, cualquier orden
# - "subset": el agente llamó AL MENOS las tools esperadas
# - "superset": el agente no llamó tools FUERA de las esperadas
trajectory_evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="subset",    # TODO: ¿es este el modo correcto?
    tool_args_match_mode="ignore",
)

In [ ]:
def build_reference_trajectory(question: str, expected_tools: list, expected_answer: str) -> list:
    """Construye la trayectoria de referencia para comparar."""
    traj = [{"role": "user", "content": question}]
    if expected_tools:
        traj.append({
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {"function": {"name": t, "arguments": "{}"}} for t in expected_tools
            ]
        })
        traj.append({"role": "tool", "content": "resultado"})
    traj.append({"role": "assistant", "content": expected_answer})
    return traj

## 3.4 Evaluador 4: LLM-as-judge multidimensional

TODO: Personaliza el prompt de evaluación para tu dominio.
El prompt debe evaluar las 3 dimensiones: correctness, efficiency, faithfulness.

In [ ]:
CUSTOM_EVAL_PROMPT = """You are an expert evaluator of AI agent trajectories.
Evaluate the following agent trajectory on THREE dimensions.

<Trajectory>
{outputs}
</Trajectory>

Score each dimension from 0.0 to 1.0:

1. **Correctness**: Is the final answer factually correct and complete?
   - 1.0: Correct and complete
   - 0.5: Partially correct or incomplete
   - 0.0: Wrong or missing

2. **Efficiency**: Did the agent use the minimum tools necessary?
   - 1.0: Only called tools when truly needed
   - 0.5: Called some unnecessary tools
   - 0.0: Called tools when it could answer from knowledge, or failed to call needed tools

3. **Faithfulness**: Is the final answer grounded in the tool results?
   - 1.0: Answer is fully based on retrieved data
   - 0.5: Answer mixes retrieved data with assumptions
   - 0.0: Answer ignores tool results or hallucinates

TODO: Agrega criterios específicos para tu dominio aquí si lo necesitas.

Respond ONLY with a JSON object:
{{
  "correctness": <float 0.0-1.0>,
  "efficiency": <float 0.0-1.0>,
  "faithfulness": <float 0.0-1.0>,
  "comment": "<brief explanation in Spanish>"
}}"""

In [ ]:
from pydantic import BaseModel

In [ ]:
class EvalScores(BaseModel):
    correctness: float
    efficiency: float
    faithfulness: float
    comment: str

In [ ]:
def multidim_judge(trajectory: list) -> dict:
    """Evaluador LLM-as-judge con 3 dimensiones numéricas."""
    structured_judge = llm.with_structured_output(EvalScores)
    prompt = CUSTOM_EVAL_PROMPT.format(
        outputs=json.dumps(trajectory, indent=2, ensure_ascii=False)
    )
    result: EvalScores = structured_judge.invoke([HumanMessage(content=prompt)])
    return {
        "correctness": result.correctness,
        "efficiency": result.efficiency,
        "faithfulness": result.faithfulness,
        "comment": result.comment,
    }

In [ ]:
# Prueba el juez con una pregunta
test_result = run_agent_and_get_trajectory(eval_dataset[0]["question"])
judge_scores = multidim_judge(test_result["trajectory"])
print("Scores del juez:", judge_scores)

─────────────────────────────────────────────────────────────
PARTE 4: Loop de evaluación y análisis (20 min)
─────────────────────────────────────────────────────────────

## 4.1 Loop completo

TODO: Ajusta el sleep_between según tu rate limit
- NVIDIA NIM: 8-10 segundos
- Groq: 3-5 segundos

In [ ]:
def run_full_eval(dataset: list, sleep_between: int = 8) -> list:
    results = []

    for i, item in enumerate(dataset):
        print(f"\n[{i+1}/{len(dataset)}] {item['question'][:60]}...")
        try:
            agent_result = run_agent_and_get_trajectory(item["question"])

            # Evaluador 1: tool use
            tool_eval = evaluate_tool_use(
                agent_result["tools_called"], item["expected_tools"]
            )

            # Evaluador 2: retrieval quality
            retrieval_eval = evaluate_retrieval(
                agent_result["trajectory"], item["expected_content"]
            )

            # Evaluador 3: trajectory match
            reference = build_reference_trajectory(
                item["question"], item["expected_tools"], item["expected_answer"]
            )
            traj_eval = trajectory_evaluator(
                outputs=agent_result["trajectory"],
                reference_outputs=reference,
            )

            # Evaluador 4: LLM-as-judge
            judge_eval = multidim_judge(agent_result["trajectory"])

            retrieval_str = f"{retrieval_eval['score']:.2f}" if retrieval_eval["score"] is not None else "N/A"

            results.append({
                "question": item["question"],
                "answer": agent_result["answer"],
                "tools_called": str(agent_result["tools_called"]),
                "expected_tools": str(item["expected_tools"]),
                "tool_use_score": tool_eval["score"],
                "retrieval_score": retrieval_eval["score"],
                "trajectory_match": float(traj_eval["score"]),
                "correctness": judge_eval["correctness"],
                "efficiency": judge_eval["efficiency"],
                "faithfulness": judge_eval["faithfulness"],
                "comment": judge_eval["comment"],
                "status": "ok",
            })

            print(f"  tool_use={tool_eval['score']:.1f} | "
                  f"retrieval={retrieval_str} | "
                  f"trajectory={float(traj_eval['score']):.1f} | "
                  f"correct={judge_eval['correctness']:.1f} | "
                  f"efficient={judge_eval['efficiency']:.1f} | "
                  f"faithful={judge_eval['faithfulness']:.1f}")

        except GraphRecursionError:
            results.append({"question": item["question"], "status": "recursion_limit"})
            print("  ❌ Recursion limit")
        except Exception as e:
            if "429" in str(e):
                print(f"  ⏳ Rate limit, esperando {sleep_between * 2}s...")
                time.sleep(sleep_between * 2)
            results.append({"question": item["question"], "status": f"error: {str(e)[:80]}"})

        time.sleep(sleep_between)

    return results

In [ ]:
eval_results = run_full_eval(eval_dataset, sleep_between=8)

## 4.2 Análisis de resultados

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(eval_results)
ok_df = df[df["status"] == "ok"].copy()
retrieval_df = ok_df[ok_df["retrieval_score"].notna()]

In [ ]:
# ── Tabla de resultados ────────────────────────────────────────
det_cols   = ["question", "tool_use_score", "retrieval_score", "trajectory_match"]
judge_cols = ["question", "correctness", "efficiency", "faithfulness"]

In [ ]:
print("=" * 65)
print("EVALUADORES DETERMINÍSTICOS")
print("=" * 65)
print(ok_df[det_cols].to_string(index=False))

In [ ]:
print("\n" + "=" * 65)
print("LLM-AS-JUDGE (3 dimensiones)")
print("=" * 65)
print(ok_df[judge_cols].to_string(index=False))

In [ ]:
# ── Promedios ──────────────────────────────────────────────────
print("\n" + "=" * 65)
print("PROMEDIOS")
print("=" * 65)
print(f"  tool_use_score   : {ok_df['tool_use_score'].mean():.2f}  (n={len(ok_df)})")
if len(retrieval_df) > 0:
    print(f"  retrieval_score  : {retrieval_df['retrieval_score'].mean():.2f}  (n={len(retrieval_df)}, solo búsquedas)")
print(f"  trajectory_match : {ok_df['trajectory_match'].mean():.2f}  (n={len(ok_df)})")
print(f"  correctness      : {ok_df['correctness'].mean():.2f}  (n={len(ok_df)})")
print(f"  efficiency       : {ok_df['efficiency'].mean():.2f}  (n={len(ok_df)})")
print(f"  faithfulness     : {ok_df['faithfulness'].mean():.2f}  (n={len(ok_df)})")
print(f"\n  Exitosas: {len(ok_df)}/{len(df)} | Errores: {len(df[df['status'] != 'ok'])}")

## 4.3 Integración con LangSmith: dataset (opcional)

LangSmith organiza las evaluaciones en **datasets** — colecciones de ejemplos
con inputs y outputs de referencia. Cada ejemplo contiene:
- `inputs`: lo que recibe el agente (la pregunta)
- `outputs`: el ground truth (expected_answer, expected_tools, expected_content)

El patrón get-or-create evita duplicar ejemplos si se corre la celda varias veces.

In [ ]:
from langsmith import Client

In [ ]:
client = Client()
dataset_name = f"workshop-{os.environ.get('USER', 'student')}-evals"

In [ ]:
datasets = list(client.list_datasets(dataset_name=dataset_name))

In [ ]:
if datasets:
    dataset = datasets[0]
    print(f"Dataset existente: {dataset.name}")
else:
    dataset = client.create_dataset(dataset_name)
    for item in eval_dataset:
        client.create_example(
            inputs={"question": item["question"]},
            outputs={
                "expected_answer": item["expected_answer"],
                "expected_tools": item["expected_tools"],
                "expected_content": item["expected_content"],
            },
            dataset_id=dataset.id,
        )
    print(f"Dataset creado con {len(eval_dataset)} ejemplos")

## 4.4 Integración con LangSmith: experimento (opcional)

`client.evaluate()` corre el agente sobre cada ejemplo del dataset,
aplica los evaluadores y registra los resultados en LangSmith.

Cada llamada crea un **experimento** con nombre único (prefix + hash).
Puedes comparar múltiples experimentos en el dashboard — por ejemplo,
distintas versiones del system prompt — sobre el mismo dataset.

`max_concurrency=1` es importante para no superar los rate limits de NVIDIA NIM.

In [ ]:
def agent_target(inputs: dict) -> dict:
    result = run_agent_and_get_trajectory(inputs["question"])
    return {
        "answer": result["answer"],
        "trajectory": result["trajectory"],
        "tools_called": result["tools_called"],
    }

In [ ]:
def eval_tool_use_ls(outputs: dict, reference_outputs: dict) -> dict:
    result = evaluate_tool_use(
        outputs["tools_called"], reference_outputs["expected_tools"]
    )
    return {"key": "tool_use", "score": result["score"]}

In [ ]:
def eval_retrieval_ls(outputs: dict, reference_outputs: dict) -> dict:
    result = evaluate_retrieval(
        outputs["trajectory"], reference_outputs.get("expected_content", [])
    )
    if result["score"] is None:
        return {"key": "retrieval_quality", "score": None}
    return {"key": "retrieval_quality", "score": result["score"]}

In [ ]:
def eval_trajectory_ls(outputs: dict, reference_outputs: dict) -> dict:
    reference = build_reference_trajectory(
        outputs["trajectory"][0]["content"],
        reference_outputs["expected_tools"],
        reference_outputs["expected_answer"],
    )
    result = trajectory_evaluator(
        outputs=outputs["trajectory"], reference_outputs=reference
    )
    return {"key": "trajectory_match", "score": float(result["score"])}

In [ ]:
def eval_llm_judge_ls(outputs: dict) -> list:
    """Una sola llamada al LLM — retorna 3 scores como lista."""
    result = multidim_judge(outputs["trajectory"])
    return [
        {"key": "correctness",  "score": result["correctness"]},
        {"key": "efficiency",   "score": result["efficiency"]},
        {"key": "faithfulness", "score": result["faithfulness"]},
    ]

In [ ]:
experiment = client.evaluate(
    agent_target,
    data=dataset_name,
    evaluators=[eval_tool_use_ls, eval_retrieval_ls, eval_trajectory_ls, eval_llm_judge_ls],
    experiment_prefix="workshop",
    max_concurrency=1,
)

## 4.5 Preguntas de reflexión

Responde estas preguntas en una celda Markdown del notebook:

1. ¿Cuál fue el evaluador con el score más bajo? ¿Por qué crees que fue así?

2. ¿Hubo algún caso donde llm_judge fue alto pero los evaluadores
   determinísticos fueron bajos? ¿Qué significa eso?

3. Si retrieval_score fue bajo en alguna pregunta: ¿fue problema del
   query que construyó el LLM, o los términos en expected_content eran
   demasiado restrictivos? ¿Cómo lo distinguirías?

4. ¿Qué cambio específico harías en el system prompt para mejorar
   el score más bajo? Formula la hipótesis antes de implementarla.

5. Si retrieval_score fue alto pero faithfulness fue bajo:
   ¿qué implica eso? ¿Es un problema grave?

## 4.6 Mejora iterativa (bonus)

TODO (opcional): Identifica el evaluador con el score más bajo,
modifica el system prompt y vuelve a correr para comparar:

results_v1 = pd.DataFrame(eval_results)

SYSTEM_PROMPT = SystemMessage(content="... versión mejorada ...")
llm_with_tools = llm.bind_tools(TOOLS, parallel_tool_calls=False)
agent = graph_builder.compile()

eval_results_v2 = run_full_eval(eval_dataset, sleep_between=8)
results_v2 = pd.DataFrame(eval_results_v2)

compare_cols = ["tool_use_score", "trajectory_match", "correctness",
                "efficiency", "faithfulness"]
ok_v1 = results_v1[results_v1["status"] == "ok"]
ok_v2 = results_v2[results_v2["status"] == "ok"]
print("V1:", ok_v1[compare_cols].mean().round(2).to_dict())
print("V2:", ok_v2[compare_cols].mean().round(2).to_dict())

Esta es la esencia del ciclo de evaluation-driven development:
Evaluar → Identificar debilidades → Mejorar → Evaluar de nuevo